In [1]:
import sys
import pandas as pd
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.append(str(ROOT))
from src.split import temporal_split

ratings = pd.read_csv(ROOT / "data" / "raw" / "ratings.csv")
movies = pd.read_csv(ROOT / "data" / "raw" / "movies.csv")
train, val, test = temporal_split(ratings)

In [2]:
stats = train.groupby("movieId")["rating"].agg(count="count", mean="mean").reset_index()
C = train["rating"].mean()
m = 50
stats["score"] = (stats["count"] / (stats["count"] + m)) * stats["mean"] + (m / (stats["count"] + m)) * C

popular = stats.merge(movies, on="movieId").sort_values("score", ascending=False)
print(popular.head(10)[["title", "count", "mean", "score"]])

                                                  title  count      mean  \
266                    Shawshank Redemption, The (1994)    282  4.418440   
633                               Godfather, The (1972)    175  4.302857   
2139                                  Fight Club (1999)    198  4.262626   
215           Star Wars: Episode IV - A New Hope (1977)    237  4.223629   
1868                                 Matrix, The (1999)    261  4.199234   
856   Star Wars: Episode V - The Empire Strikes Back...    193  4.227979   
247                                 Pulp Fiction (1994)    272  4.178309   
46                           Usual Suspects, The (1995)    185  4.218919   
302                                 Forrest Gump (1994)    304  4.156250   
491                    Silence of the Lambs, The (1991)    253  4.169960   

         score  
266   4.282242  
633   4.127575  
2139  4.111711  
215   4.100015  
1868  4.089081  
856   4.081088  
247   4.075169  
46    4.068954  
302   4.06

In [3]:
def top_by_genre(genre, n=10):
    g = popular[popular["genres"].str.contains(genre, regex=False)]
    return g.head(n)[["movieId", "title", "genres", "count", "score"]]

print(top_by_genre("Sci-Fi"))
print(top_by_genre("Comedy"))

      movieId                                              title  \
215       260          Star Wars: Episode IV - A New Hope (1977)   
1868     2571                                 Matrix, The (1999)   
856      1196  Star Wars: Episode V - The Empire Strikes Back...   
869      1210  Star Wars: Episode VI - Return of the Jedi (1983)   
4622     7361       Eternal Sunshine of the Spotless Mind (2004)   
456       541                                Blade Runner (1982)   
6618    79132                                   Inception (2010)   
927      1270                          Back to the Future (1985)   
31         32          Twelve Monkeys (a.k.a. 12 Monkeys) (1995)   
6128    60069                                      WALL·E (2008)   

                                               genres  count     score  
215                           Action|Adventure|Sci-Fi    237  4.100015  
1868                           Action|Sci-Fi|Thriller    261  4.089081  
856                           Ac

In [4]:
for name, part in [("Validation", val), ("Test", test)]:
    absent = ~part["movieId"].isin(train["movieId"])
    print(f"{name}: {absent.mean():.1%} des notes sur des films absents de Train "
          f"({part.loc[absent, 'movieId'].nunique()} films)")

Validation: 7.4% des notes sur des films absents de Train (694 films)
Test: 9.5% des notes sur des films absents de Train (872 films)


**Interprétation** : écris la part des films de Validation et de Test absents de Train, et explique que ce cold start justifie l'approche hybride.